---

# Tugas Besar 2 IF3170: Prediksi Pembatalan Pemesanan Penginapan

*Eksperimen `knn_booking_month`: `knn_selection_distance` ditambah satu fitur, bulan pemesanan.*

## Ringkasan

Sama dengan `knn_selection_distance` (public 0,85505): 8 fitur, standardisasi, Manhattan, distance voting `1/d`. Perubahannya hanya satu fitur baru:

- `booking_month`: bulan saat tamu memesan, dari tanggal kedatangan dikurangi `days_before_arrival`

Dari 9 fitur baru yang diuji (hitungan grup, kepadatan hari, harga relatif, total biaya, hari pemesanan, porsi akhir pekan, dan lainnya), hanya fitur ini yang membantu: rata-rata seed 1 sampai 5 0,8432 vs 0,8418, menang di 5 dari 5 seed. k 31 dipilih dari seed 42.

**Distance-weighted voting (di luar materi).** Suara tetangga dibobot `1 / (d + 1e-6)`, sehingga tetangga yang lebih dekat lebih menentukan.

## Import Libraries

In [ ]:
import numpy as np
import pandas as pd

## Settings

In [ ]:
class Settings:
    SEED = 42
    TARGET = 'is_canceled'
    ID_COLUMN = 'id'
    FEATURES = ['days_before_arrival', 'arrival_index', 'special_requests', 'nightly_rate_idr',
                'booking_channel', 'arrival_dayofyear', 'total_nights', 'needs_parking', 'booking_month']
    K = 31
    P = 1
    SUB_PATH = 'submission.csv'

np.random.seed(Settings.SEED)

## Load Dataset

Isi variabel `train` (dengan kolom `is_canceled`) dan `test` sebagai DataFrame. Kolom `id` dibaca sebagai teks.

In [ ]:
train = pd.read_csv('train.csv', dtype={'id': str})
test = pd.read_csv('test.csv', dtype={'id': str})

pd.Series({'train': len(train), 'test': len(test)})

## Kode Pendukung

Preprocessing bersama kelompok dan fungsi validasi.

In [ ]:
TARGET = 'is_canceled'


ID_COL = 'id'


CAT_COLS = ['meal_package', 'room_category', 'booking_channel']


# Buang baris duplikat persis (fitur dan target). Hanya untuk data train.
def clean_train(df):
    feature_cols = [c for c in df.columns if c != ID_COL]
    return df.drop_duplicates(subset=feature_cols).reset_index(drop=True)


# Ubah nilai yang tidak masuk akal (num_adults = 0) menjadi missing.
def mark_invalid(df):
    df = df.copy()
    df.loc[df['num_adults'] == 0, 'num_adults'] = np.nan
    return df


# Tambah fitur turunan per baris, aman dipakai pada train dan test secara terpisah.
def add_features(df):
    df = df.copy()
    df['total_nights'] = df['weekend_nights'] + df['weekday_nights']
    df['total_guests'] = df['num_adults'] + df['num_children']
    df['is_free'] = (df['nightly_rate_idr'] == 0).astype(float)
    df.loc[df['nightly_rate_idr'].isna(), 'is_free'] = np.nan
    df['rate_per_guest'] = df['nightly_rate_idr'] / df['total_guests'].replace(0, np.nan)

    arrival = pd.to_datetime(
        dict(year=df['arrival_year'], month=df['arrival_month'], day=df['arrival_day']),
        errors='coerce',
    )
    df['arrival_weekday'] = arrival.dt.day_of_week.astype(float)
    df['arrival_dayofyear'] = arrival.dt.day_of_year.astype(float)
    df['arrival_index'] = ((arrival - pd.Timestamp('2023-01-01')).dt.days).astype(float)
    return df

In [ ]:
# Confusion matrix biner dalam bentuk [[TN, FP], [FN, TP]].
def confusion_matrix(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    return np.array([[tn, fp], [fn, tp]])


# F1 untuk kelas 0 dan kelas 1, hasilnya [f1_0, f1_1].
def f1_per_class(y_true, y_pred):
    (tn, fp), (fn, tp) = confusion_matrix(y_true, y_pred)
    f1_pos = 2 * tp / (2 * tp + fp + fn) if tp + fp + fn > 0 else 0.0
    f1_neg = 2 * tn / (2 * tn + fn + fp) if tn + fn + fp > 0 else 0.0
    return np.array([f1_neg, f1_pos])


# Macro F1, rata-rata F1 kedua kelas (metrik resmi kompetisi).
def macro_f1(y_true, y_pred):
    return float(f1_per_class(y_true, y_pred).mean())


# Cari threshold dengan Macro F1 tertinggi, dipakai pada prediksi out-of-fold.
def best_threshold(y_true, proba, grid=None):
    if grid is None:
        grid = np.round(np.arange(0.20, 0.81, 0.01), 2)
    scores = np.array([macro_f1(y_true, (proba >= t).astype(int)) for t in grid])
    i = int(np.argmax(scores))
    return float(grid[i]), float(scores[i])


# Bagi indeks menjadi k fold dengan proporsi kelas sama, hasilnya list (train_idx, val_idx).
def stratified_kfold(y, n_splits=5, seed=42):
    y = np.asarray(y)
    rng = np.random.RandomState(seed)
    fold_of = np.empty(len(y), dtype=int)
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        fold_of[idx] = np.arange(len(idx)) % n_splits
    return [(np.where(fold_of != k)[0], np.where(fold_of == k)[0]) for k in range(n_splits)]

## Preprocessing

`engineer` menambah fitur turunan kelompok dan bulan pemesanan (`booking_month`).

`prepare` melakukan fit pada `fit_df` lalu mengubah semua frame: missing numerik diisi median, standardisasi `(x - mean) / std`, `booking_channel` di-one-hot dengan nilai 0,5 (jarak kategori beda = 1). Hasilnya array float32 dengan urutan kolom `FEATURES`.

In [ ]:
def engineer(df):
    E = add_features(mark_invalid(df))
    E['booking_index'] = E['arrival_index'] - E['days_before_arrival']
    E['booking_month'] = (pd.Timestamp('2023-01-01') + pd.to_timedelta(E['booking_index'], unit='D')).dt.month
    return E


def prepare(fit_df, *frames):
    fit_e, frames_e = engineer(fit_df), [engineer(f) for f in frames]
    out = [[] for _ in frames_e]
    for f in Settings.FEATURES:
        if f in CAT_COLS:
            mode = fit_e[f].mode()[0]
            for level in sorted(fit_e[f].dropna().unique()):
                for o, e in zip(out, frames_e):
                    o.append(((e[f].fillna(mode) == level) * 0.5).to_numpy())
            continue
        median = fit_e[f].median()
        filled = fit_e[f].fillna(median)
        mean, std = filled.mean(), filled.std()
        for o, e in zip(out, frames_e):
            o.append(((e[f].fillna(median) - mean) / std).to_numpy())
    return [np.column_stack(o).astype(np.float32) for o in out]

## KNN from Scratch

- `fit` menyimpan data train
- `kneighbors` menghitung jarak Minkowski pangkat p per kolom lalu mengambil k terdekat, per potongan 500 query agar memori cukup
- `predict_proba` memakai bobot `1 / d`

In [ ]:
class KNN:

    def __init__(self, k=31, p=1, batch_size=500):
        self.k, self.p, self.batch_size = k, p, batch_size

    def fit(self, X, y):
        self.X_, self.y_ = X, np.asarray(y).astype(int)
        return self

    def kneighbors(self, Q):
        all_d, all_i = [], []
        for s in range(0, len(Q), self.batch_size):
            q = Q[s:s + self.batch_size]
            total = np.zeros((len(q), len(self.X_)), np.float32)
            for j in range(self.X_.shape[1]):
                total += np.abs(q[:, j, None] - self.X_[None, :, j]) ** self.p
            idx = np.argpartition(total, self.k - 1, axis=1)[:, :self.k]
            d = np.take_along_axis(total, idx, 1)
            order = np.argsort(d, axis=1, kind='stable')
            all_i.append(np.take_along_axis(idx, order, 1))
            all_d.append(np.take_along_axis(d, order, 1) ** (1 / self.p))
        return np.vstack(all_d), np.vstack(all_i)

    def predict_proba(self, Q):
        d, idx = self.kneighbors(Q)
        w = 1.0 / (d + 1e-6)
        return (w * self.y_[idx]).sum(1) / w.sum(1)

## Validasi dan Threshold

Stratified 5-fold seed 42. Preprocessing di-fit ulang pada train fold. Threshold diambil dari OOF.

In [ ]:
train = clean_train(train)
y = train[Settings.TARGET].to_numpy()
folds = stratified_kfold(y, n_splits=5, seed=Settings.SEED)

oof = np.zeros(len(y))
for tr, va in folds:
    X_tr, X_va = prepare(train.iloc[tr], train.iloc[tr], train.iloc[va])
    oof[va] = KNN(Settings.K, Settings.P).fit(X_tr, y[tr]).predict_proba(X_va)

THRESHOLD, score = best_threshold(y, oof)
print('OOF macro F1', round(score, 4), ' threshold', THRESHOLD)

## Submission

Preprocessing dan KNN di-fit dengan seluruh data train.

In [ ]:
X_full, X_test = prepare(train, train, test)
proba = KNN(Settings.K, Settings.P).fit(X_full, y).predict_proba(X_test)

submission = pd.DataFrame({Settings.ID_COLUMN: test[Settings.ID_COLUMN], Settings.TARGET: (proba >= THRESHOLD).astype(int)})
submission.to_csv(Settings.SUB_PATH, index=False)
submission[Settings.TARGET].value_counts(normalize=True).round(4)